In [1]:
from torchvision.models import resnet18
from torchvision import datasets,transforms
import torch.nn as nn,torch
from torch.utils.data import DataLoader
train_transform = transforms.Compose([
    transforms.RandomCrop(32,padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])
train_dataset = datasets.CIFAR10(
    root="../CIFAR-10/data",
    train=True,
    download=False,
    transform=train_transform
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True   
)
test_dataset = datasets.CIFAR10(
    root="../CIFAR-10/data",
    train=False,
    download=False,
    transform=test_transform
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=256,    
)

In [2]:
model = resnet18(weights="DEFAULT")
model.fc = nn.Linear(512,10)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=1e-3
)
device = torch.device("cuda")
model = model.to(device)

In [3]:
for epoch in range(25):
    total = 0
    correct = 0
    model.train()
    for images,labels in train_dataloader:
        model.zero_grad()
        images = images.to(device)
        labels = labels.to(device)
        out = model(images)
        loss = criterion(out,labels)
        loss.backward()
        optimizer.step()
        out = out.argmax(dim=1)
        correct += (out==labels).sum().item()
        total += labels.size(0)
    print(f"Train Accuracy:{correct/total*100}")


Train Accuracy:64.86
Train Accuracy:76.106
Train Accuracy:79.042
Train Accuracy:81.184
Train Accuracy:82.14399999999999
Train Accuracy:83.54599999999999
Train Accuracy:84.11
Train Accuracy:85.088
Train Accuracy:85.718
Train Accuracy:86.42999999999999
Train Accuracy:86.642
Train Accuracy:87.376
Train Accuracy:87.77199999999999
Train Accuracy:88.5
Train Accuracy:88.51
Train Accuracy:88.794
Train Accuracy:89.548
Train Accuracy:89.526
Train Accuracy:89.974
Train Accuracy:90.218
Train Accuracy:90.49000000000001
Train Accuracy:90.702
Train Accuracy:90.86800000000001
Train Accuracy:91.262
Train Accuracy:91.5


In [5]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images,labels in test_dataloader:
        images= images.to(device)
        labels= labels.to(device)
        out = model(images)
        out = out.argmax(dim=1)
        correct += (out==labels).sum().item()
        total += labels.size(0)
    print(f"Test Accuracy :{correct/total*100}")

Test Accuracy :85.35000000000001
